# 12. Build Song Harmonic Similarity Index

This notebook builds the missing song-level data layer for harmonic recommendation and exploration.

Earlier notebooks created global exact and harmonic vocabularies (`V_n`, `H_n`) plus genre/decade document-term tables. For song-to-song similarity, we need the object that was intentionally postponed:

```text
song_harmonic_terms(song_id, n, harmonic_id, count)
```

That table is the sparse representation of each song's weighted multiset `H_n(s)`. Once it exists, we can compute song-level document frequency, TF-IDF weights, and explainable nearest-neighbor results.


## Design

The full harmonic vocabulary has more than 20 million classes. A full song-by-`H_n` matrix is possible but large, and it would include many one-off parse artifacts that are not helpful for a first interactive prototype.

This notebook therefore builds a **support-limited but configurable** song index:

1. Select harmonic classes with global support `count >= MIN_GLOBAL_HARMONIC_COUNT`.
2. Stream each song's chord sequence.
3. Convert chord windows to transposition-invariant `harmonic_id` values using the same helpers as notebook 2.
4. Count supported harmonic classes per song and `n`.
5. Compute song-level document frequency and IDF.
6. Build TF-IDF weights.
7. Prototype explainable similarity search.

The threshold is not a mathematical assumption. It is an indexing parameter. Lower it to surface rarer fingerprints, raise it for faster iteration.


## Setup


In [1]:
from collections import Counter
from functools import lru_cache
from pathlib import Path
import importlib
import os
import sys
import time

import duckdb
import pandas as pd
from tqdm import tqdm

CWD = Path.cwd()
ROOT = CWD.parent if (CWD / "utils").exists() else CWD
MPLCONFIGDIR = ROOT / ".matplotlib-cache"
MPLCONFIGDIR.mkdir(exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(MPLCONFIGDIR))

NOTEBOOK_DIR = ROOT / "notebooks"
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

from utils import duckdb_store as ds
from utils import ngram_features as nf

importlib.reload(ds)
importlib.reload(nf)

RAW_PATH = ROOT / "data" / "raw" / "chordonomicon_v2.csv"
DB_PATH = ROOT / "data" / "processed" / "harmonic_trends.duckdb"

assert RAW_PATH.exists(), RAW_PATH
assert DB_PATH.exists(), DB_PATH

print({
    "root": str(ROOT),
    "raw_path": str(RAW_PATH),
    "db_path": str(DB_PATH),
    "duckdb_version": duckdb.__version__,
    "ngram_features": str(Path(nf.__file__).resolve()),
})


{'root': '/Users/juansalinas/Documents/GitHub/harmonic-trends', 'raw_path': '/Users/juansalinas/Documents/GitHub/harmonic-trends/data/raw/chordonomicon_v2.csv', 'db_path': '/Users/juansalinas/Documents/GitHub/harmonic-trends/data/processed/harmonic_trends.duckdb', 'duckdb_version': '1.5.2', 'ngram_features': '/Users/juansalinas/Documents/GitHub/harmonic-trends/notebooks/utils/ngram_features.py'}


## Parameters

`MIN_GLOBAL_HARMONIC_COUNT=100` keeps roughly 160k harmonic classes in the current database. That is broad enough to include many distinctive motifs while avoiding millions of singleton classes.

For quick development, set `MAX_SONGS` to a small number such as `25_000`. For the production index, leave it as `None`.

This notebook writes derived tables into the DuckDB database. It does not modify the existing global vocabulary tables, but `REBUILD_TABLES=True` will replace any prior song-similarity index tables with the same names.

`HARMONIC_ID_CACHE_SIZE` keeps repeated exact windows from recomputing the same harmonic id. It is bounded so the full run does not try to cache the entire exact n-gram vocabulary.


In [2]:
NS = tuple(range(3, 9))
MIN_GLOBAL_HARMONIC_COUNT = 100
CHUNKSIZE = 25_000
FLUSH_ROWS = 500_000
HARMONIC_ID_CACHE_SIZE = 1_000_000
MAX_SONGS = None  # e.g. 25_000 for a smoke run
REBUILD_TABLES = True

TERM_TABLE = "song_harmonic_terms"
TOTAL_TABLE = "song_harmonic_totals"
DF_TABLE = "harmonic_song_document_frequency"
TFIDF_TABLE = "song_harmonic_tfidf"
NORM_TABLE = "song_harmonic_norm_components"
INDEX_METADATA_TABLE = "song_harmonic_similarity_index_metadata"

DEFAULT_N_WEIGHTS = {n: n / min(NS) for n in NS}
DEFAULT_N_WEIGHTS


{3: 1.0,
 4: 1.3333333333333333,
 5: 1.6666666666666667,
 6: 2.0,
 7: 2.3333333333333335,
 8: 2.6666666666666665}

## Preflight

Inspect the candidate vocabulary size at several support thresholds before building the index.


In [3]:
con = duckdb.connect(str(DB_PATH), read_only=True)
ds.configure_connection(con)
try:
    threshold_audit = []
    total_windows_by_n = con.execute(
        """
        SELECT n, SUM(count) AS total_windows
        FROM harmonic_ngrams
        GROUP BY n
        """
    ).fetchdf()

    for threshold in [10, 25, 50, 100, 250, 500, 1_000, 2_500, 5_000]:
        rows = con.execute(
            """
            SELECT
                n,
                COUNT(*) AS harmonic_classes,
                SUM(count) AS supported_windows
            FROM harmonic_ngrams
            WHERE count >= ?
            GROUP BY n
            ORDER BY n
            """,
            [threshold],
        ).fetchdf()
        rows.insert(0, "min_global_count", threshold)
        rows = rows.merge(total_windows_by_n, on="n", how="left")
        rows["global_window_coverage"] = rows["supported_windows"] / rows["total_windows"]
        threshold_audit.append(rows)
    threshold_audit = pd.concat(threshold_audit, ignore_index=True)
finally:
    con.close()

class_audit = threshold_audit.pivot(
    index="min_global_count", columns="n", values="harmonic_classes"
).assign(total=lambda df: df.sum(axis=1))

coverage_audit = threshold_audit.pivot(
    index="min_global_count", columns="n", values="global_window_coverage"
).assign(mean_coverage=lambda df: df.mean(axis=1))

class_audit, coverage_audit


(n                     3       4       5       6       7       8    total
 min_global_count                                                        
 10                51790  139241  228413  300517  352554  379643  1452158
 25                27683   64020   96157  119747  134663  136837   579107
 50                17111   36058   51678   63265   68638   66875   303625
 100               10637   21054   28930   34254   35395   33225   163495
 250                5680   10392   13516   15224   14649   13186    72647
 500                3532    5915    7599    8191    7346    6361    38944
 1000               2239    3428    4275    4283    3693    3262    21180
 2500               1150    1748    2029    1703    1500    1212     9342
 5000                684    1031     996     873     718     576     4878,
 n                        3         4         5         6         7         8  \
 min_global_count                                                               
 10                0.99


The preflight returns two tables:

- `class_audit`: how many harmonic classes would be indexed at each support threshold.
- `coverage_audit`: the share of global harmonic windows covered by those classes.

This matters because storage is driven by song-feature rows, while musical recall is driven by window coverage. A lower threshold improves recall but can materially increase build time and database size.


## Load Support-Limited Harmonic Vocabulary

The in-memory lookup is `set[harmonic_id]` per `n`. We compute harmonic ids directly from chord windows rather than joining through `exact_to_harmonic`, because the direct computation preserves the same normalization logic while avoiding a 34M-row lookup table in memory.


In [4]:
def load_supported_harmonic_ids(db_path: Path, min_global_count: int, ns: tuple[int, ...]) -> dict[int, set[str]]:
    con = duckdb.connect(str(db_path), read_only=True)
    ds.configure_connection(con)
    try:
        vocab = con.execute(
            """
            SELECT n, harmonic_id
            FROM harmonic_ngrams
            WHERE n IN (SELECT * FROM UNNEST(?))
              AND count >= ?
            """,
            [list(ns), min_global_count],
        ).fetchdf()
    finally:
        con.close()

    return {int(n): set(group["harmonic_id"]) for n, group in vocab.groupby("n")}

supported_harmonic_ids = load_supported_harmonic_ids(DB_PATH, MIN_GLOBAL_HARMONIC_COUNT, NS)
{n: len(ids) for n, ids in supported_harmonic_ids.items()}


{3: 10637, 4: 21054, 5: 28930, 6: 34254, 7: 35395, 8: 33225}

## Streaming Term Builder

This is the main indexing pass. It scans song chord strings, counts supported harmonic classes per song, and writes the sparse terms into DuckDB.

Output:

```text
song_harmonic_terms(song_id, n, harmonic_id, count)
```


In [5]:
@lru_cache(maxsize=HARMONIC_ID_CACHE_SIZE)
def cached_harmonic_id(ngram: tuple[str, ...]) -> str:
    return nf.harmonic_id(nf.first_root_normalized_key(ngram))


def song_harmonic_counts(tokens: list[str], ns: tuple[int, ...], supported: dict[int, set[str]]) -> list[dict]:
    rows = []
    for n in ns:
        allowed = supported.get(n, set())
        if not allowed or len(tokens) < n:
            continue

        counter = Counter()
        for ngram in nf.chord_ngrams(tokens, n):
            hid = cached_harmonic_id(ngram)
            if hid in allowed:
                counter[hid] += 1

        for harmonic_id, count in counter.items():
            rows.append({"n": n, "harmonic_id": harmonic_id, "count": int(count)})
    return rows


def flush_term_rows(con: duckdb.DuckDBPyConnection, rows: list[dict]) -> int:
    if not rows:
        return 0
    df = pd.DataFrame(rows)
    con.register("term_rows_df", df)
    con.execute(f"INSERT INTO {TERM_TABLE} SELECT song_id, n, harmonic_id, count FROM term_rows_df")
    con.unregister("term_rows_df")
    return len(df)


def raw_song_count(raw_path: Path) -> int:
    # Counts data rows without loading chord strings into memory.
    with raw_path.open("rb") as f:
        return max(sum(1 for _ in f) - 1, 0)


def build_song_harmonic_terms(
    *,
    db_path: Path,
    raw_path: Path,
    ns: tuple[int, ...],
    supported: dict[int, set[str]],
    chunksize: int,
    flush_rows: int,
    max_songs: int | None,
    rebuild: bool,
) -> dict:
    con = duckdb.connect(str(db_path))
    ds.configure_connection(con)
    started = time.time()
    buffer = []
    songs_seen = 0
    rows_written = 0
    flushes = 0
    total_songs = raw_song_count(raw_path)
    progress_total = min(total_songs, max_songs) if max_songs is not None else total_songs

    try:
        if rebuild:
            con.execute(f"DROP TABLE IF EXISTS {TERM_TABLE}")

        con.execute(
            f"""
            CREATE TABLE IF NOT EXISTS {TERM_TABLE} (
                song_id BIGINT,
                n INTEGER,
                harmonic_id VARCHAR,
                count INTEGER
            )
            """
        )

        reader = pd.read_csv(raw_path, usecols=["id", "chords"], chunksize=chunksize)
        with tqdm(total=progress_total, desc="songs indexed", unit="song") as pbar:
            for chunk in reader:
                for row in chunk.itertuples(index=False):
                    song_id = int(row.id)
                    tokens = nf.chord_tokens(row.chords)
                    for term in song_harmonic_counts(tokens, ns, supported):
                        term["song_id"] = song_id
                        buffer.append(term)

                    songs_seen += 1
                    pbar.update(1)
                    if songs_seen % 1_000 == 0:
                        cache_info = cached_harmonic_id.cache_info()
                        pbar.set_postfix(
                            indexed_rows=rows_written,
                            buffered=len(buffer),
                            flushes=flushes,
                            cache_hits=cache_info.hits,
                            cache_size=cache_info.currsize,
                        )

                    if len(buffer) >= flush_rows:
                        rows_written += flush_term_rows(con, buffer)
                        flushes += 1
                        buffer.clear()
                        cache_info = cached_harmonic_id.cache_info()
                        pbar.set_postfix(
                            indexed_rows=rows_written,
                            buffered=0,
                            flushes=flushes,
                            cache_hits=cache_info.hits,
                            cache_size=cache_info.currsize,
                        )

                    if max_songs is not None and songs_seen >= max_songs:
                        break

                if max_songs is not None and songs_seen >= max_songs:
                    break

        rows_written += flush_term_rows(con, buffer)
        if buffer:
            flushes += 1
        buffer.clear()

        with tqdm(total=2, desc="term indexes", unit="step") as pbar:
            con.execute(f"CREATE INDEX IF NOT EXISTS {TERM_TABLE}_song_idx ON {TERM_TABLE}(song_id)")
            pbar.update(1)
            con.execute(f"CREATE INDEX IF NOT EXISTS {TERM_TABLE}_feature_idx ON {TERM_TABLE}(n, harmonic_id)")
            pbar.update(1)

        elapsed = time.time() - started
        cache_info = cached_harmonic_id.cache_info()
        summary = {
            "songs_seen": songs_seen,
            "rows_written": rows_written,
            "flushes": flushes,
            "elapsed_seconds": elapsed,
            "songs_per_second": songs_seen / elapsed if elapsed else None,
            "rows_per_second": rows_written / elapsed if elapsed else None,
            "harmonic_id_cache_hits": cache_info.hits,
            "harmonic_id_cache_misses": cache_info.misses,
            "harmonic_id_cache_currsize": cache_info.currsize,
        }
        return summary
    finally:
        con.close()


In [6]:
if REBUILD_TABLES:
    build_summary = build_song_harmonic_terms(
        db_path=DB_PATH,
        raw_path=RAW_PATH,
        ns=NS,
        supported=supported_harmonic_ids,
        chunksize=CHUNKSIZE,
        flush_rows=FLUSH_ROWS,
        max_songs=MAX_SONGS,
        rebuild=True,
    )
else:
    build_summary = {"skipped": True}

build_summary


term indexes: 100%|██████████| 2/2 [25:20<00:00, 760.23s/step]


{'songs_seen': 679807,
 'rows_written': 66950002,
 'flushes': 134,
 'elapsed_seconds': 7817.115734100342,
 'songs_per_second': 86.96391650369212,
 'rows_per_second': 8564.540206043803,
 'harmonic_id_cache_hits': 236069938,
 'harmonic_id_cache_misses': 57552692,
 'harmonic_id_cache_currsize': 1000000}

## Build Totals, Document Frequency, And TF-IDF

`song_harmonic_totals` uses all harmonic windows from `song_ngram_summary`, not only windows that survived the support threshold. This keeps term-frequency normalization honest.

`harmonic_song_document_frequency` uses song-level document frequency, which is the key statistic for rare-fingerprint discovery.

`song_harmonic_norm_components` stores per-song/per-`n` squared-vector norms so interactive similarity queries do not have to rescan the full TF-IDF table just to normalize cosine scores.



Storage choice: `song_harmonic_terms` is the canonical sparse count table. `song_harmonic_tfidf` intentionally materializes derived weights, even though those weights could be computed as a view, because interactive nearest-neighbor queries should not repeatedly join counts, totals, and IDF. DuckDB column compression should keep the repeated numeric columns manageable, and the raw count table remains available for auditing or rebuilding weights.


In [7]:
def build_similarity_weight_tables(db_path: Path) -> None:
    con = duckdb.connect(str(db_path))
    ds.configure_connection(con)
    phase_total = 12
    try:
        with tqdm(total=phase_total, desc="weight tables", unit="step") as pbar:
            pbar.set_postfix(step="drop totals")
            con.execute(f"DROP TABLE IF EXISTS {TOTAL_TABLE}")
            pbar.update(1)

            pbar.set_postfix(step="create totals")
            con.execute(
                f"""
                CREATE TABLE {TOTAL_TABLE} AS
                SELECT id AS song_id, 3 AS n, n_ngrams_3 AS total_windows FROM song_ngram_summary
                UNION ALL SELECT id, 4, n_ngrams_4 FROM song_ngram_summary
                UNION ALL SELECT id, 5, n_ngrams_5 FROM song_ngram_summary
                UNION ALL SELECT id, 6, n_ngrams_6 FROM song_ngram_summary
                UNION ALL SELECT id, 7, n_ngrams_7 FROM song_ngram_summary
                UNION ALL SELECT id, 8, n_ngrams_8 FROM song_ngram_summary
                """
            )
            pbar.update(1)

            pbar.set_postfix(step="add coverage")
            con.execute(
                f"""
                CREATE OR REPLACE TABLE {TOTAL_TABLE} AS
                SELECT
                    totals.song_id,
                    totals.n,
                    totals.total_windows,
                    COALESCE(term_counts.unique_harmonic_classes, 0) AS unique_indexed_harmonic_classes,
                    COALESCE(term_counts.indexed_windows, 0) AS indexed_windows,
                    COALESCE(term_counts.indexed_windows, 0)::DOUBLE / NULLIF(totals.total_windows, 0) AS indexed_window_coverage
                FROM {TOTAL_TABLE} totals
                LEFT JOIN (
                    SELECT song_id, n, COUNT(*) AS unique_harmonic_classes, SUM(count) AS indexed_windows
                    FROM {TERM_TABLE}
                    GROUP BY song_id, n
                ) term_counts USING (song_id, n)
                """
            )
            pbar.update(1)

            pbar.set_postfix(step="index totals")
            con.execute(f"CREATE INDEX IF NOT EXISTS {TOTAL_TABLE}_song_n_idx ON {TOTAL_TABLE}(song_id, n)")
            pbar.update(1)

            pbar.set_postfix(step="drop df")
            con.execute(f"DROP TABLE IF EXISTS {DF_TABLE}")
            pbar.update(1)

            pbar.set_postfix(step="create df/idf")
            con.execute(
                f"""
                CREATE TABLE {DF_TABLE} AS
                WITH song_counts AS (
                    SELECT n, COUNT(*) AS n_songs
                    FROM {TOTAL_TABLE}
                    WHERE total_windows > 0
                    GROUP BY n
                ), df AS (
                    SELECT n, harmonic_id, COUNT(*) AS song_df
                    FROM {TERM_TABLE}
                    GROUP BY n, harmonic_id
                )
                SELECT
                    df.n,
                    df.harmonic_id,
                    df.song_df,
                    song_counts.n_songs,
                    df.song_df::DOUBLE / song_counts.n_songs AS song_df_rate,
                    LN((song_counts.n_songs + 1)::DOUBLE / (df.song_df + 1)) + 1 AS idf,
                    h.count AS global_count,
                    h.frequency AS global_frequency,
                    h.example_ngram,
                    h.harmonic_key,
                    h.n_exact_ngrams
                FROM df
                JOIN song_counts USING (n)
                LEFT JOIN harmonic_ngrams h USING (n, harmonic_id)
                """
            )
            pbar.update(1)

            pbar.set_postfix(step="index df")
            con.execute(f"CREATE INDEX IF NOT EXISTS {DF_TABLE}_feature_idx ON {DF_TABLE}(n, harmonic_id)")
            pbar.update(1)

            pbar.set_postfix(step="drop tfidf")
            con.execute(f"DROP TABLE IF EXISTS {TFIDF_TABLE}")
            pbar.update(1)

            pbar.set_postfix(step="create tfidf")
            con.execute(
                f"""
                CREATE TABLE {TFIDF_TABLE} AS
                SELECT
                    t.song_id,
                    t.n,
                    t.harmonic_id,
                    t.count,
                    totals.total_windows,
                    t.count::DOUBLE / NULLIF(totals.total_windows, 0) AS tf_frequency,
                    LN(1 + t.count) AS tf_log_count,
                    df.idf,
                    (t.count::DOUBLE / NULLIF(totals.total_windows, 0)) * df.idf AS tfidf_frequency,
                    LN(1 + t.count) * df.idf AS tfidf_log_count
                FROM {TERM_TABLE} t
                JOIN {TOTAL_TABLE} totals USING (song_id, n)
                JOIN {DF_TABLE} df USING (n, harmonic_id)
                """
            )
            pbar.update(1)

            pbar.set_postfix(step="index tfidf")
            con.execute(f"CREATE INDEX IF NOT EXISTS {TFIDF_TABLE}_song_idx ON {TFIDF_TABLE}(song_id)")
            con.execute(f"CREATE INDEX IF NOT EXISTS {TFIDF_TABLE}_feature_idx ON {TFIDF_TABLE}(n, harmonic_id)")
            pbar.update(1)

            pbar.set_postfix(step="create norms")
            con.execute(f"DROP TABLE IF EXISTS {NORM_TABLE}")
            con.execute(
                f"""
                CREATE TABLE {NORM_TABLE} AS
                SELECT
                    song_id,
                    n,
                    SUM(tfidf_frequency * tfidf_frequency) AS norm_sq_tfidf_frequency,
                    SUM(tfidf_log_count * tfidf_log_count) AS norm_sq_tfidf_log_count
                FROM {TFIDF_TABLE}
                GROUP BY song_id, n
                """
            )
            con.execute(f"CREATE INDEX IF NOT EXISTS {NORM_TABLE}_song_n_idx ON {NORM_TABLE}(song_id, n)")
            pbar.update(1)

            pbar.set_postfix(step="metadata")
            con.execute(f"DROP TABLE IF EXISTS {INDEX_METADATA_TABLE}")
            con.execute(
                f"""
                CREATE TABLE {INDEX_METADATA_TABLE} AS
                SELECT * FROM (VALUES
                    ('version', 'song_harmonic_similarity_index_v1'),
                    ('min_global_harmonic_count', CAST(? AS VARCHAR)),
                    ('ns', ?),
                    ('max_songs', ?),
                    ('term_weight_default', 'tfidf_log_count'),
                    ('idf_policy', 'song-level idf: ln((N_n + 1) / (df_nh + 1)) + 1'),
                    ('n_weight_default', 'alpha_n = n / 3'),
                    ('norm_policy', 'precomputed per-song/per-n squared norms; query applies alpha_n^2'),
                    ('harmonic_id_cache_size', CAST(? AS VARCHAR))
                ) AS metadata(key, value)
                """,
                [MIN_GLOBAL_HARMONIC_COUNT, ",".join(map(str, NS)), str(MAX_SONGS), HARMONIC_ID_CACHE_SIZE],
            )
            pbar.update(1)
            pbar.set_postfix(step="done")
    finally:
        con.close()


build_similarity_weight_tables(DB_PATH)


weight tables: 100%|██████████| 12/12 [00:45<00:00,  3.78s/step, step=done]         


## Validate Index Tables


In [8]:
con = duckdb.connect(str(DB_PATH), read_only=True)
ds.configure_connection(con)
try:
    table_counts = con.execute(
        f"""
        SELECT '{TERM_TABLE}' AS table_name, COUNT(*) AS rows FROM {TERM_TABLE}
        UNION ALL SELECT '{TOTAL_TABLE}', COUNT(*) FROM {TOTAL_TABLE}
        UNION ALL SELECT '{DF_TABLE}', COUNT(*) FROM {DF_TABLE}
        UNION ALL SELECT '{TFIDF_TABLE}', COUNT(*) FROM {TFIDF_TABLE}
        UNION ALL SELECT '{NORM_TABLE}', COUNT(*) FROM {NORM_TABLE}
        """
    ).fetchdf()

    coverage = con.execute(
        f"""
        SELECT
            n,
            COUNT(*) AS song_n_rows,
            AVG(indexed_window_coverage) AS avg_indexed_window_coverage,
            MEDIAN(indexed_window_coverage) AS median_indexed_window_coverage,
            AVG(unique_indexed_harmonic_classes) AS avg_unique_indexed_classes
        FROM {TOTAL_TABLE}
        WHERE total_windows > 0
        GROUP BY n
        ORDER BY n
        """
    ).fetchdf()

    idf_sample = con.execute(
        f"""
        SELECT n, harmonic_id, example_ngram, song_df, song_df_rate, idf, global_count, global_frequency
        FROM {DF_TABLE}
        ORDER BY idf DESC, global_count DESC
        LIMIT 20
        """
    ).fetchdf()
finally:
    con.close()

table_counts, coverage, idf_sample


(                         table_name      rows
 0               song_harmonic_terms  66950002
 1              song_harmonic_totals   4078842
 2  harmonic_song_document_frequency    163495
 3               song_harmonic_tfidf  66950002
 4     song_harmonic_norm_components   3899667,
    n  song_n_rows  avg_indexed_window_coverage  \
 0  3       679623                     0.971199   
 1  4       679483                     0.908211   
 2  5       679193                     0.821162   
 3  6       678488                     0.723667   
 4  7       677638                     0.623249   
 5  8       676677                     0.531842   
 
    median_indexed_window_coverage  avg_unique_indexed_classes  
 0                        1.000000                   16.222856  
 1                        1.000000                   18.625370  
 2                        0.978261                   18.607807  
 3                        0.868421                   17.218033  
 4                        0.70833

## Explainable TF-IDF Cosine Similarity

The production prototype should score with selected `n` values and adjustable length weights.

For a song `s`, feature `(n,h)` has default weight:

```text
x_s,n,h = alpha_n * log(1 + count_s,n,h) * idf_n,h
```

Similarity is cosine similarity:

```text
sim(s,t) = <x_s, x_t> / (||x_s|| ||x_t||)
```

The notebook precomputes per-song/per-`n` squared norms, so sliders can change `alpha_n` without rescanning the entire TF-IDF table for normalization. Each shared harmonic feature contributes:

```text
contribution_n,h = x_s,n,h * x_t,n,h / (||x_s|| ||x_t||)
```


In [9]:
def _weight_frame(n_weights: dict[int, float]) -> pd.DataFrame:
    return pd.DataFrame(
        [{"n": int(n), "alpha": float(alpha)} for n, alpha in n_weights.items()]
    )


def _norm_column(weight_column: str) -> str:
    if weight_column == "tfidf_log_count":
        return "norm_sq_tfidf_log_count"
    if weight_column == "tfidf_frequency":
        return "norm_sq_tfidf_frequency"
    raise ValueError("weight_column must be 'tfidf_log_count' or 'tfidf_frequency'")


def similar_songs(
    query_song_id: int,
    *,
    db_path: Path = DB_PATH,
    selected_ns: tuple[int, ...] = NS,
    n_weights: dict[int, float] | None = None,
    top_k: int = 20,
    min_shared_features: int = 2,
    weight_column: str = "tfidf_log_count",
) -> pd.DataFrame:
    if n_weights is None:
        n_weights = {n: DEFAULT_N_WEIGHTS[n] for n in selected_ns}
    else:
        n_weights = {n: n_weights[n] for n in selected_ns}

    norm_column = _norm_column(weight_column)
    alpha_df = _weight_frame(n_weights)
    con = duckdb.connect(str(db_path), read_only=True)
    ds.configure_connection(con)
    con.register("selected_n_weights", alpha_df)
    try:
        result = con.execute(
            f"""
            WITH q AS (
                SELECT
                    w.song_id,
                    w.n,
                    w.harmonic_id,
                    w.{weight_column} * a.alpha AS weight
                FROM {TFIDF_TABLE} w
                JOIN selected_n_weights a USING (n)
                WHERE w.song_id = ?
            ), candidate_dots AS (
                SELECT
                    c.song_id AS candidate_song_id,
                    SUM(q.weight * c.{weight_column} * a.alpha) AS dot_product,
                    COUNT(*) AS shared_features
                FROM q
                JOIN {TFIDF_TABLE} c USING (n, harmonic_id)
                JOIN selected_n_weights a ON a.n = c.n
                WHERE c.song_id <> ?
                GROUP BY c.song_id
                HAVING COUNT(*) >= ?
            ), norms AS (
                SELECT
                    nc.song_id,
                    SQRT(SUM(nc.{norm_column} * a.alpha * a.alpha)) AS norm
                FROM {NORM_TABLE} nc
                JOIN selected_n_weights a USING (n)
                GROUP BY nc.song_id
            ), q_norm AS (
                SELECT norm FROM norms WHERE song_id = ?
            )
            SELECT
                d.candidate_song_id,
                d.dot_product / NULLIF(q_norm.norm * norms.norm, 0) AS similarity_score,
                d.dot_product,
                d.shared_features,
                q_norm.norm AS query_norm,
                norms.norm AS candidate_norm,
                m.release_year,
                m.decade,
                m.main_genre,
                m.artist_id,
                m.spotify_song_id
            FROM candidate_dots d
            JOIN norms ON norms.song_id = d.candidate_song_id
            CROSS JOIN q_norm
            LEFT JOIN song_metadata m ON m.id = d.candidate_song_id
            ORDER BY similarity_score DESC, shared_features DESC, candidate_song_id
            LIMIT ?
            """,
            [query_song_id, query_song_id, min_shared_features, query_song_id, top_k],
        ).fetchdf()
    finally:
        con.unregister("selected_n_weights")
        con.close()
    return result


def similarity_explanation(
    query_song_id: int,
    candidate_song_id: int,
    *,
    db_path: Path = DB_PATH,
    selected_ns: tuple[int, ...] = NS,
    n_weights: dict[int, float] | None = None,
    weight_column: str = "tfidf_log_count",
    top_features: int = 40,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    if n_weights is None:
        n_weights = {n: DEFAULT_N_WEIGHTS[n] for n in selected_ns}
    else:
        n_weights = {n: n_weights[n] for n in selected_ns}

    norm_column = _norm_column(weight_column)
    alpha_df = _weight_frame(n_weights)
    con = duckdb.connect(str(db_path), read_only=True)
    ds.configure_connection(con)
    con.register("selected_n_weights", alpha_df)
    try:
        feature_contrib = con.execute(
            f"""
            WITH norms AS (
                SELECT
                    nc.song_id,
                    SQRT(SUM(nc.{norm_column} * a.alpha * a.alpha)) AS norm
                FROM {NORM_TABLE} nc
                JOIN selected_n_weights a USING (n)
                WHERE nc.song_id IN (?, ?)
                GROUP BY nc.song_id
            ), shared AS (
                SELECT
                    q.n,
                    q.harmonic_id,
                    df.example_ngram,
                    df.song_df,
                    df.song_df_rate,
                    df.idf,
                    q.count AS query_count,
                    c.count AS candidate_count,
                    q.{weight_column} * a.alpha AS query_weight,
                    c.{weight_column} * a.alpha AS candidate_weight,
                    q.{weight_column} * c.{weight_column} * a.alpha * a.alpha AS raw_contribution
                FROM {TFIDF_TABLE} q
                JOIN {TFIDF_TABLE} c USING (n, harmonic_id)
                JOIN selected_n_weights a USING (n)
                JOIN {DF_TABLE} df USING (n, harmonic_id)
                WHERE q.song_id = ?
                  AND c.song_id = ?
            ), denom AS (
                SELECT q.norm * c.norm AS norm_product
                FROM norms q
                JOIN norms c ON c.song_id = ?
                WHERE q.song_id = ?
            )
            SELECT
                shared.*,
                shared.raw_contribution / NULLIF(denom.norm_product, 0) AS cosine_contribution
            FROM shared
            CROSS JOIN denom
            ORDER BY cosine_contribution DESC, n DESC, harmonic_id
            LIMIT ?
            """,
            [query_song_id, candidate_song_id, query_song_id, candidate_song_id, candidate_song_id, query_song_id, top_features],
        ).fetchdf()

        by_n = con.execute(
            f"""
            WITH norms AS (
                SELECT
                    nc.song_id,
                    SQRT(SUM(nc.{norm_column} * a.alpha * a.alpha)) AS norm
                FROM {NORM_TABLE} nc
                JOIN selected_n_weights a USING (n)
                WHERE nc.song_id IN (?, ?)
                GROUP BY nc.song_id
            ), shared AS (
                SELECT q.n, q.{weight_column} * c.{weight_column} * a.alpha * a.alpha AS raw_contribution
                FROM {TFIDF_TABLE} q
                JOIN {TFIDF_TABLE} c USING (n, harmonic_id)
                JOIN selected_n_weights a USING (n)
                WHERE q.song_id = ?
                  AND c.song_id = ?
            ), denom AS (
                SELECT q.norm * c.norm AS norm_product
                FROM norms q
                JOIN norms c ON c.song_id = ?
                WHERE q.song_id = ?
            )
            SELECT
                n,
                COUNT(*) AS shared_features,
                SUM(raw_contribution) AS raw_contribution,
                SUM(raw_contribution) / NULLIF(MAX(norm_product), 0) AS cosine_contribution
            FROM shared
            CROSS JOIN denom
            GROUP BY n
            ORDER BY n
            """,
            [query_song_id, candidate_song_id, query_song_id, candidate_song_id, candidate_song_id, query_song_id],
        ).fetchdf()
    finally:
        con.unregister("selected_n_weights")
        con.close()

    return feature_contrib, by_n


## Smoke Query

Pick a query song that has indexed terms, then inspect its nearest neighbors and strongest shared harmonic evidence.


In [10]:
con = duckdb.connect(str(DB_PATH), read_only=True)
ds.configure_connection(con)
try:
    query_song_id = con.execute(
        f"""
        SELECT song_id
        FROM {TFIDF_TABLE}
        GROUP BY song_id
        ORDER BY COUNT(*) DESC, song_id
        LIMIT 1
        """
    ).fetchone()[0]
finally:
    con.close()

query_song_id


426642

In [11]:
neighbors = similar_songs(query_song_id, top_k=10, min_shared_features=4)
neighbors


,candidate_song_id,similarity_score,dot_product,shared_features,query_norm,candidate_norm,release_year,decade,main_genre,artist_id,spotify_song_id
0,372474,0.305324,105937.154620,147,1125.7698,308.203582,2015,2010,pop,artist_1327,5a7NdkF09AfD0H607eiOkX
1,427145,0.268901,90397.060908,92,1125.7698,298.615903,2015,2010,pop,artist_1327,0PaemlGLiH2O6nAxRCzmee
2,427019,0.265290,67404.409627,131,1125.7698,225.693024,2015,2010,pop,artist_1327,7bbmEmHMczUCXsGx3QQgmg
3,427021,0.263720,87458.626541,92,1125.7698,294.584229,2015,2010,pop,artist_1327,4elU36XAVPHYReonz6qqcR
4,427015,0.260586,84392.209922,170,1125.7698,287.675260,2015,2010,pop,artist_1327,2m3Fxma8rzIbHTnsq8Hpip
5,427880,0.258256,80430.702090,192,1125.7698,276.644315,2015,2010,pop,artist_1327,3R2bkLqP197fl3vKHpONrg
6,103779,0.253437,72631.610010,56,1125.7698,254.568831,<NA>,<NA>,None,None,None
7,273720,0.244635,50290.646419,56,1125.7698,182.607593,<NA>,<NA>,None,artist_65539,None
8,429745,0.241319,76792.410158,87,1125.7698,282.668577,<NA>,<NA>,None,None,None
9,201616,0.240380,65168.627756,67,1125.7698,240.819180,<NA>,<NA>,rock,artist_5228,5f3KIOH87rvFwfiY098IrY


In [12]:
if not neighbors.empty:
    candidate_song_id = int(neighbors.iloc[0]["candidate_song_id"])
    feature_evidence, n_breakdown = similarity_explanation(query_song_id, candidate_song_id)
else:
    candidate_song_id = None
    feature_evidence = pd.DataFrame()
    n_breakdown = pd.DataFrame()

candidate_song_id, n_breakdown, feature_evidence.head(20)


(372474,
    n  shared_features  raw_contribution  cosine_contribution
 0  3               18       1241.616339             0.003578
 1  4               22       3873.000307             0.011162
 2  5               26       9264.193872             0.026701
 3  6               29      18112.393876             0.052202
 4  7               26      28955.134270             0.083452
 5  8               26      44490.815956             0.128228,
     n          harmonic_id                example_ngram  song_df  \
 0   8  H8_f21edf7154cb1e4a     D Emin C D Emin Amin D C       78   
 1   8  H8_723e695b5db24502     Bmin G A Bmin Emin A G A       58   
 2   8  H8_83043ea08f960278     D Emin Amin D C D Emin C       64   
 3   8  H8_b44b8a83c115f42d     Emin Amin D C D Emin C D      102   
 4   8  H8_0674bb0dd1a6cc56     C D Emin Amin D C D Emin      105   
 5   8  H8_3ff4e19a080a5317     C D Emin C D Emin Amin D      384   
 6   8  H8_6985a3da88048ba7     Dmin G F G Amin F G Amin      226   
 7  

## Next Architecture Step

Once the smoke query looks musically reasonable, move the retrieval functions into a small reusable module and build an API around these operations:

- search/select song by local id, Spotify id, artist id, genre, or enriched title metadata;
- retrieve neighbors under selected `n` values and `alpha_n` weights;
- return per-feature and per-`n` explanations;
- compare rankings from fixed-`n` and multi-`n` settings.

The app should remain an explorer of harmonic evidence, not a black-box recommender.
